# Throughput optimization kernels — Colab test

Tests **fused MoE expert**, **fused QKV**, **fused QKVG**, and **fused O+gate** (correctness and optional speed).

**On Colab:** Run the **Install (P4de-matching env)** cell first (PyTorch 2.10 has a memory leak; we use 2.7.1+cu128 like P4de), then **Restart runtime**, then run path setup and test cells.

**Note:** Fused MoE expert test needs `grouped_gemm`; if missing, that test is skipped. Other tests only need torch + triton.\n\n**T4 / 64 KB shared memory:** Colab’s T4 has a 64 KB per-block shared-memory limit. The kernels detect this and automatically use smaller block sizes (so you may see slightly lower throughput on Colab; full throughput is on P4de/A100).

## Install (P4de-matching env)

**Option A (recommended): uv** — Uses `pyproject.toml` + `uv.lock` so torch 2.7.1+cu128 stays pinned. Run the uv cell, restart runtime, then run path + test cells.

**Option B: pip** — Single `pip install` with both indexes so the resolver doesn't pull torch 2.10. Restart runtime after install.

In [ ]:
# === Option A: uv (recommended; lockfile keeps torch 2.7.1+cu128) ===
# PROJECT_DIR = folder with pyproject.toml and uv.lock (e.g. Drive or /content)
import os
PROJECT_DIR = '/content/drive/MyDrive/Test20_3B_Zero3'  # change if your repo is elsewhere
if not os.path.isdir(PROJECT_DIR) or not os.path.isfile(os.path.join(PROJECT_DIR, 'uv.lock')):
    cwd = os.getcwd()
    PROJECT_DIR = os.path.dirname(cwd) if os.path.basename(cwd) == 'notebooks' else cwd
    if not os.path.isfile(os.path.join(PROJECT_DIR, 'uv.lock')):
        PROJECT_DIR = '/content/Test20_3B_Zero3'
PROJECT_DIR = os.path.abspath(PROJECT_DIR)
os.chdir(PROJECT_DIR)
!pip install uv -q
!uv sync
print(f'uv sync done in {PROJECT_DIR}. Restart runtime, then run path + test cells.')

# === Option B: pip (single command so resolver does not pull torch 2.10) ===
# Uncomment and run if you prefer pip. Restart runtime after.
# !pip uninstall -y torch torchvision torchaudio 2>/dev/null || true
# !pip install torch==2.7.1+cu128 torchvision==0.22.1+cu128 triton==3.6.0 deepspeed==0.18.6 einops==0.8.2 transformers "datasets>=2.14.0" "accelerate>=0.24.0" "safetensors>=0.7.0" "tqdm>=4.65.0" boto3 pyyaml huggingface_hub "numpy>=1.24.0" "pandas>=2.0.0" ninja --extra-index-url https://download.pytorch.org/whl/cu128
# print('Done. Restart runtime.')

In [ ]:
# Optional: full P4de stack (for training; needed for fused_moe_expert correctness test vs grouped_gemm)
# Run after the main install cell (torch must be installed first).
# grouped-gemm setup.py calls torch.cuda.get_device_capability() during metadata unless we set TORCH_CUDA_ARCH_LIST:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"   # T4; use "8.0" for A100, "8.9" for L4
os.environ["GROUPED_GEMM_CUTLASS"] = "1"     # optional: CUTLASS kernels (faster)
!pip install "setuptools<81"
!pip install grouped-gemm==0.3.0 --no-build-isolation
# !pip install "flash-linear-attention @ git+https://github.com/sustcsonglin/flash-linear-attention@d792da6dd544d28980205817b20535500c3f243e"

In [ ]:
# Path to Test20_3B_Zero3. Set UV_PROJECT_DIR if you ran uv sync elsewhere.
import os
import sys

UV_PROJECT_DIR = '/content/drive/MyDrive/Test20_3B_Zero3'  # where you ran uv sync (has .venv, pyproject.toml, uv.lock)
COLAB_ROOT = None

cwd = os.getcwd()
if COLAB_ROOT and os.path.isdir(COLAB_ROOT):
    base = COLAB_ROOT
else:
    base = os.path.dirname(cwd) if os.path.basename(cwd) == 'notebooks' else cwd

test20 = os.path.join(base, 'experiments', 'tests', 'Test20_3B_Zero3')
if os.path.isdir(test20):
    TEST_ROOT = test20
elif os.path.isdir(os.path.join(base, 'code', 'src')):
    TEST_ROOT = base
else:
    TEST_ROOT = base

# Find uv .venv (must run before import torch). Try UV_PROJECT_DIR first, then common locations.
py_ver = f"python{sys.version_info.major}.{sys.version_info.minor}"
candidates = [UV_PROJECT_DIR, TEST_ROOT, '/content/drive/MyDrive/Test20_3B_Zero3', '/content/Test20_3B_Zero3', '/content/LLM/experiments/tests/Test20_3B_Zero3', cwd]
venv_site = None
for proj in candidates:
    if proj and os.path.isdir(proj):
        v = os.path.join(proj, '.venv', 'lib', py_ver, 'site-packages')
        if os.path.isdir(v):
            venv_site = v
            break
if venv_site and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
    print(f'Using uv venv: {venv_site}')
elif not venv_site and (UV_PROJECT_DIR or os.path.isdir(os.path.join(TEST_ROOT, 'uv.lock'))):
    print('uv .venv not found. Set UV_PROJECT_DIR to the folder where you ran uv sync (e.g. /content/Test20_3B_Zero3).')

CODE_DIR = os.path.join(TEST_ROOT, "code")
SRC_DIR = os.path.join(CODE_DIR, "src")
# SRC_DIR on path -> "from kernels.fused_moe_expert import ..." resolves to code/src/kernels/
if os.path.isdir(SRC_DIR) and SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"TEST_ROOT: {TEST_ROOT}")
print(f"SRC_DIR exists: {os.path.isdir(SRC_DIR)}")

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    major, minor = torch.cuda.get_device_capability(0)
    print(f"  arch: {major}.{minor}  -> TORCH_CUDA_ARCH_LIST="{major}{minor}" for grouped-gemm")
    try:
        p = torch.cuda.get_device_properties(0)
        mem_gb = getattr(p, 'total_memory', getattr(p, 'total_mem', 0)) / 1e9
        print(f"VRAM: {mem_gb:.1f} GB")
    except Exception:
        pass
else:
    if torch.__version__.startswith('2.10'):
        print('CUDA is missing: Colab often keeps torch 2.10. Run the Install cell, then Runtime → Restart runtime.')
    else:
        print('CUDA not available. Ensure you restarted runtime after installing torch+cu128.')

try:
    import triton
    print(f"Triton: {triton.__version__}")
except ImportError:
    print("Triton not available")

## Correctness tests

If you see **PTXASError** or **ptxas failed with error code -9** (SIGKILL), the JIT compiler was likely killed by Colab (OOM). Restart runtime, run only the path cell and this cell. Triton cache is set to `/tmp` to reduce memory pressure; tests that hit ptxas failure are skipped with a message.

In [ ]:
# Triton: use local disk for cache (avoids Drive I/O); free memory before JIT
import os
import sys
import gc
os.environ.setdefault("TRITON_CACHE_DIR", "/tmp/triton_colab")
# Force small blocks on 64 KB shared-memory GPUs (T4) so our Triton kernels don't hit OutOfResources
if torch.cuda.is_available():
    try:
        p = torch.cuda.get_device_properties(0)
        limit = getattr(p, "max_shared_memory_per_block", 0) or getattr(p, "shared_memory_per_block", 0) or 0
        name = getattr(p, "name", "") or ""
        if (limit > 0 and limit <= 65536) or "T4" in name:
            os.environ["TRITON_USE_SMALL_BLOCKS"] = "1"
            print("T4/64KB GPU: TRITON_USE_SMALL_BLOCKS=1 set.")
    except Exception:
        pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
# Fix Triton backend binaries when venv is on Google Drive (Drive drops execute bits)
try:
    for _path in (os.path.join(p, 'triton', 'backends', 'nvidia', 'bin') for p in sys.path if p and os.path.isdir(p)):
        if os.path.isdir(_path):
            for _f in os.listdir(_path):
                os.chmod(os.path.join(_path, _f), 0o755)
            break
except Exception as _e:
    print("Note: could not chmod triton bin:", _e)

def _is_ptxas_failure(e):
    s = str(e).lower() + type(e).__name__
    return "ptxas" in s or "sigkill" in s or "-9" in s or "error code" in s

def _is_shared_mem_oom(e):
    s = str(e).lower()
    return "outofresources" in s or ("shared memory" in s and "65536" in s)

def test_fused_moe_expert():
    try:
        from kernels.fused_moe_expert import fused_moe_expert_forward
    except ImportError:
        print("SKIP fused_moe_expert (kernels not importable)")
        return True
    try:
        from kernels.moe_grouped_gemm import moe_grouped_gemm
    except ImportError:
        print("SKIP fused_moe_expert (grouped_gemm not available)")
        return True
    if not torch.cuda.is_available():
        print("SKIP fused_moe_expert (no CUDA)")
        return True
    # grouped_gemm backend only supports bfloat16
    E, D, H = 4, 128, 64
    m_sizes = torch.tensor([16, 24, 8, 32], device="cuda")
    expert_counts = m_sizes
    N = int(m_sizes.sum().item())
    dtype = torch.bfloat16
    x = torch.randn(N, D, device="cuda", dtype=dtype) * 0.02
    W_gate = torch.randn(E, D, H, device="cuda", dtype=dtype) * 0.02
    W_up = torch.randn(E, D, H, device="cuda", dtype=dtype) * 0.02
    W_down = torch.randn(E, H, D, device="cuda", dtype=dtype) * 0.02
    out_fused = fused_moe_expert_forward(x, W_gate, W_up, W_down, expert_counts, use_triton=False)
    try:
        gate_ref = moe_grouped_gemm(x, W_gate, expert_counts)
    up_ref = moe_grouped_gemm(x, W_up, expert_counts)
    h_ref = torch.nn.functional.silu(gate_ref) * up_ref
    out_ref = moe_grouped_gemm(h_ref, W_down, expert_counts)
    torch.testing.assert_close(out_fused.float(), out_ref.float(), atol=1e-2, rtol=1e-2)
    print("PASS fused_moe_expert")
    return True

def test_fused_qkv():
    try:
        from kernels.fused_qkv_proj import fused_qkv_proj_forward
    except ImportError:
        print("SKIP fused_qkv (kernels not importable)")
        return True
    if not torch.cuda.is_available():
        return True
    try:
        N, D = 64, 128
        x = torch.randn(N, D, device="cuda", dtype=torch.float32) * 0.02
        W_q = torch.randn(D, D, device="cuda", dtype=torch.float32) * 0.02
        W_k = torch.randn(D, D, device="cuda", dtype=torch.float32) * 0.02
        W_v = torch.randn(D, D, device="cuda", dtype=torch.float32) * 0.02
        q_f, k_f, v_f = fused_qkv_proj_forward(x, W_q, W_k, W_v)
        q_ref = torch.nn.functional.linear(x, W_q)
        k_ref = torch.nn.functional.linear(x, W_k)
        v_ref = torch.nn.functional.linear(x, W_v)
        torch.testing.assert_close(q_f.float(), q_ref.float(), atol=1e-4, rtol=1e-3)
        torch.testing.assert_close(k_f.float(), k_ref.float(), atol=1e-4, rtol=1e-3)
        torch.testing.assert_close(v_f.float(), v_ref.float(), atol=1e-4, rtol=1e-3)
        print("PASS fused_qkv")
        return True
    except Exception as e:
        if _is_ptxas_failure(e):
            print("SKIP fused_qkv (ptxas killed, often OOM on Colab. Restart runtime, run only this notebook.)")
            return True
        raise

def test_fused_qkvg():
    try:
        from kernels.fused_qkv_proj import fused_qkvg_proj_forward
    except ImportError:
        print("SKIP fused_qkvg (kernels not importable)")
        return True
    if not torch.cuda.is_available():
        return True
    try:
        N, D_in, D_out = 64, 256, 128
        x = torch.randn(N, D_in, device="cuda", dtype=torch.float32) * 0.02
        W_q = torch.randn(D_out, D_in, device="cuda", dtype=torch.float32) * 0.02
        W_k = torch.randn(D_out, D_in, device="cuda", dtype=torch.float32) * 0.02
        W_v = torch.randn(D_out, D_in, device="cuda", dtype=torch.float32) * 0.02
        W_g = torch.randn(D_out, D_in, device="cuda", dtype=torch.float32) * 0.02
        q_f, k_f, v_f, g_f = fused_qkvg_proj_forward(x, W_q, W_k, W_v, W_g)
        for name, f, ref in [("q", q_f, torch.nn.functional.linear(x, W_q)), ("k", k_f, torch.nn.functional.linear(x, W_k)), ("v", v_f, torch.nn.functional.linear(x, W_v)), ("g", g_f, torch.nn.functional.linear(x, W_g))]:
            torch.testing.assert_close(f.float(), ref.float(), atol=1e-4, rtol=1e-3)
        print("PASS fused_qkvg")
        return True
    except Exception as e:
        if _is_ptxas_failure(e):
            print("SKIP fused_qkvg (ptxas killed, often OOM. Restart runtime, run only this notebook.)")
            return True
        raise

def test_fused_o_gate():
    try:
        from kernels.fused_qkv_proj import fused_o_gate_proj_forward
    except ImportError:
        print("SKIP fused_o_gate (kernels not importable)")
        return True
    if not torch.cuda.is_available():
        return True
    try:
        N, D = 64, 128
        x = torch.randn(N, D, device="cuda", dtype=torch.float32) * 0.02
        o_sparse = torch.randn(N, D, device="cuda", dtype=torch.float32) * 0.02
        W_go = torch.randn(D, D, device="cuda", dtype=torch.float32) * 0.02
        W_o = torch.randn(D, D, device="cuda", dtype=torch.float32) * 0.02
        out_f = fused_o_gate_proj_forward(x, o_sparse, W_go, W_o)
        g_o = torch.sigmoid(torch.nn.functional.linear(x, W_go))
        out_ref = torch.nn.functional.linear(o_sparse * g_o, W_o)
        torch.testing.assert_close(out_f.float(), out_ref.float(), atol=1e-4, rtol=1e-3)
        print("PASS fused_o_gate")
        return True
    except Exception as e:
        if _is_ptxas_failure(e):
            print("SKIP fused_o_gate (ptxas killed, often OOM. Restart runtime, run only this notebook.)")
            return True
        raise

print("=== Throughput kernels correctness ===")
test_fused_moe_expert()
test_fused_qkv()
test_fused_qkvg()
test_fused_o_gate()
print("Done.")

## Optional: quick speed check (CUDA only)

In [ ]:
if torch.cuda.is_available():
    import time
    try:
        from kernels.fused_qkv_proj import fused_qkv_proj_forward
        N, D = 256, 512
        x = torch.randn(N, D, device="cuda", dtype=torch.float16) * 0.02
        W_q = torch.randn(D, D, device="cuda", dtype=torch.float16) * 0.02
        W_k = torch.randn(D, D, device="cuda", dtype=torch.float16) * 0.02
        W_v = torch.randn(D, D, device="cuda", dtype=torch.float16) * 0.02
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(50):
            fused_qkv_proj_forward(x, W_q, W_k, W_v)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        print(f"Fused QKV 50 iters: {(t1-t0)*1000:.2f} ms ({(t1-t0)/50*1000:.2f} ms/iter)")
    except Exception as e:
        print(f"Speed check skipped: {e}")
else:
    print("No CUDA — speed check skipped.")